In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [4]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

agent_tools = Tools()
agent_tools.add_tool(search)

In [5]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [6]:
agent_tools.get_tools()


[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [13]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()
from openai import OpenAI
import os
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="openai/gpt-oss-120b",client=openai_client)
)

In [14]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


/workspaces/llm-zoomcamp-2026-code/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [15]:
result.cost


In [16]:
result.all_messages


[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01ktzbqrrffabt9hwejnrxrad9', summary=[], type='reasoning', content=[Content(text='We need to answer: "How do I run Olama locally?" Likely refers to a software named Olama. Could be a machine learning model? Let\'s search.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query

In [17]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


In [18]:
runner.run()


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


...

-> Response received


-> Response received


Chat ended.


/workspaces/llm-zoomcamp-2026-code/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:184: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  combined_cost = self.pricing_config.calculate_cost(


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='hello', role='user', phase=None, type=None), ResponseReasoningItem(id='resp_01ktzbtpbne1h93m8mx85r0psy', summary=[], type='reasoning', content=[Content(text='The user just says "hello". As a teaching assistant, respond politely, ask how can I help. No need to search.', type='reasoning_text')], encrypted_content=None, status='completed'), ResponseOutputMessage(id='msg_01ktzbtpbne1hs2aymqaw5m5q0', content=[Respon